# Infra-FM: Experiment Grid

Runs a systematic ablation over pretraining configurations, then evaluates
each with linear probe and fine-tuned classification. Produces a full
comparison table at the end.

**Grid dimensions:**
- Pretraining epochs: 25, 100
- Temperature: 0.1, 0.2
- Projection dim: 64, 128

**For each pretraining config, runs:**
1. Linear probe (frozen encoder)
2. Fine-tuned (unfrozen encoder)

Plus one random-init baseline for comparison.

**Before running:**
- Runtime → Change runtime type → GPU (T4)
- Drive should have: `infra_fm/datasets/dataset_central-america_stac_v1/`
- Drive should have: `infra_fm/code/infra_fm_curation.zip`

## 1. Mount Drive + check GPU

In [3]:
from google.colab import drive
drive.mount('/content/drive')

import torch
print(f'GPU available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('WARNING: No GPU found. Go to Runtime -> Change runtime type -> GPU.')

Mounted at /content/drive
GPU available: False


## 2. Install dependencies

In [4]:
%%capture
!pip install scipy opencv-python-headless

## 3. Extract code + set up imports

In [5]:
import os, sys, zipfile
from pathlib import Path

DRIVE_ROOT = '/content/drive/MyDrive/infra_fm'
CODE_ZIP   = f'{DRIVE_ROOT}/code/infra_fm_curation.zip'
EXTRACT_TO = '/content/infra_fm_clean'
CODE_ROOT  = f'{EXTRACT_TO}/infra_fm_code_only'

# Only extract if not already done (saves time on re-runs)
if not Path(f'{CODE_ROOT}/downstream').exists():
    print('Extracting code...')
    os.makedirs(EXTRACT_TO, exist_ok=True)
    with zipfile.ZipFile(CODE_ZIP, 'r') as z:
        for member in z.namelist():
            clean_path = member.replace('\\', '/')
            target = os.path.join(EXTRACT_TO, clean_path)
            if clean_path.endswith('/'):
                os.makedirs(target, exist_ok=True)
            else:
                os.makedirs(os.path.dirname(target), exist_ok=True)
                with z.open(member) as src, open(target, 'wb') as dst:
                    dst.write(src.read())
    print('Extraction complete.')
else:
    print('Code already extracted.')

sys.path.insert(0, CODE_ROOT)
os.chdir(CODE_ROOT)
print(f'Working directory: {os.getcwd()}')

# Verify imports
from downstream.common.models import EncoderBackbone, SimCLRModel, LinearClassifier
from downstream.common.transforms import build_eval_transform
from downstream.common.utils import choose_device, ensure_dir, save_checkpoint, save_json, set_seed
from downstream.asset_classification.datasets import AssetClassificationDataset
print('All imports OK')

Extracting code...
Extraction complete.
Working directory: /content/infra_fm_clean/infra_fm_code_only
All imports OK


## 4. Configuration — edit before running

In [6]:
import json

# --- Paths ---
DATASET_ROOT = f'{DRIVE_ROOT}/datasets/dataset_central-america_stac_v1'
GRID_DIR     = f'{DRIVE_ROOT}/results/experiment_grid'
os.makedirs(GRID_DIR, exist_ok=True)

# --- Shared training settings ---
BAND_INDICES      = '0,1,2,3,4,5,6,7,8,9'  # 10 bands
BACKBONE          = 'resnet18'
N_OPTICAL         = 7                        # sentinel2_ms bands
CLASSIFY_EPOCHS   = 30
CLASSIFY_BATCH    = 32
CLASSIFY_LR       = 1e-3
CLASSIFY_LR_PROBE = 1e-2                     # higher LR for linear probe only
WEIGHT_DECAY      = 1e-4
TRAIN_FRACTION    = 0.8
SEED              = 42
TAIL_EPOCHS       = 5
IMAGE_SIZE        = 224
PRETRAIN_BATCH    = 32
PRETRAIN_LR       = 3e-4

# --- Experiment grid ---
# Each entry defines one pretraining configuration.
# Add or remove entries to expand/shrink the grid.
PRETRAIN_GRID = [
    {"epochs": 25,  "temperature": 0.2, "projection_dim": 128, "name": "ep25_t02_pd128"},
    {"epochs": 100, "temperature": 0.2, "projection_dim": 128, "name": "ep100_t02_pd128"},
    {"epochs": 100, "temperature": 0.1, "projection_dim": 128, "name": "ep100_t01_pd128"},
    {"epochs": 100, "temperature": 0.2, "projection_dim": 64,  "name": "ep100_t02_pd64"},
]

# Verify dataset
manifest = json.load(open(f'{DATASET_ROOT}/manifest.json'))
sample_shape = manifest['records'][0].get('image_shape', [])
print(f'Dataset: {manifest["n_tiles"]} tiles')
print(f'Modalities: {manifest.get("modalities")}')
print(f'Sample shape (C,H,W): {sample_shape}')
print(f'\nGrid: {len(PRETRAIN_GRID)} pretraining configs x 2 classifiers = {len(PRETRAIN_GRID)*2} runs')
print(f'Plus 1 random-init baseline = {len(PRETRAIN_GRID)*2 + 1} total experiments')
print(f'\nGrid configs:')
for g in PRETRAIN_GRID:
    print(f"  {g['name']:25s} epochs={g['epochs']:>3}  temp={g['temperature']}  proj_dim={g['projection_dim']}")

Dataset: 1782 tiles
Modalities: ['sentinel1', 'sentinel2_ms', 'landsat_thermal']
Sample shape (C,H,W): [10, 61, 61]

Grid: 4 pretraining configs x 2 classifiers = 8 runs
Plus 1 random-init baseline = 9 total experiments

Grid configs:
  ep25_t02_pd128            epochs= 25  temp=0.2  proj_dim=128
  ep100_t02_pd128           epochs=100  temp=0.2  proj_dim=128
  ep100_t01_pd128           epochs=100  temp=0.1  proj_dim=128
  ep100_t02_pd64            epochs=100  temp=0.2  proj_dim=64


## 5. Shared utilities

In [7]:
import csv, random, time
import torch
from torch import nn
from torch.optim import AdamW
from torch.utils.data import DataLoader, Subset

# Pretraining imports
sys.path.insert(0, f'{CODE_ROOT}/curation')
from pretraining.augmentations import TwoCropTransform, build_simclr_transform
from pretraining.losses import nt_xent_loss
from pretraining.datasets import InfrastructureImageDataset


def split_indices(n, train_fraction, seed):
    idxs = list(range(n))
    random.Random(seed).shuffle(idxs)
    cut = max(1, int(n * train_fraction))
    return idxs[:cut], idxs[cut:]


def compute_class_weights(dataset, train_indices, device):
    counts = [0] * len(dataset.label_space.classes)
    for idx in train_indices:
        counts[dataset[idx]['label'].item()] += 1
    n_classes = len(counts)
    total = sum(counts)
    weights = [total / (n_classes * c) if c > 0 else 0.0 for c in counts]
    return torch.tensor(weights, dtype=torch.float32, device=device)


def evaluate(encoder, head, loader, device, classes=None):
    if not loader.dataset:
        return 0.0, None
    encoder.eval(); head.eval()
    correct, total = 0, 0
    cc = [0] * len(classes) if classes else None
    ct = [0] * len(classes) if classes else None
    with torch.no_grad():
        for batch in loader:
            x = batch['image'].to(device)
            y = batch['label'].to(device)
            pred = head(encoder(x)).argmax(dim=1)
            correct += (pred == y).sum().item()
            total   += y.numel()
            if classes:
                for i in range(len(classes)):
                    mask = (y == i)
                    cc[i] += (pred[mask] == y[mask]).sum().item()
                    ct[i] += mask.sum().item()
    overall   = correct / max(total, 1)
    per_class = {cls: round(cc[i] / max(ct[i], 1), 4)
                 for i, cls in enumerate(classes)} if classes else None
    return overall, per_class


def short_class(cls):
    """Makes class names unambiguous for display."""
    return (
        cls.replace('energy.', '')
           .replace('distribution.', 'dx_')
           .replace('transmission.', 'tx_')
    )


def run_pretraining(cfg, dataset_root, band_indices, n_optical,
                    backbone, batch_size, lr, seed, device, output_dir):
    """Runs one pretraining configuration. Returns path to best.pt."""
    out = ensure_dir(output_dir)
    best_pt = out / 'best.pt'

    # Skip if already done
    if best_pt.exists():
        print(f'  [skip] checkpoint already exists: {best_pt}')
        return str(best_pt)

    set_seed(seed)
    in_channels = len([int(x) for x in band_indices.split(',')])

    transform = TwoCropTransform(
        build_simclr_transform(224, n_optical=n_optical)
    )
    dataset = InfrastructureImageDataset(
        dataset_root=dataset_root,
        transform=transform,
        band_indices=band_indices,
    )
    loader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                        drop_last=True, num_workers=2)

    model = SimCLRModel(
        backbone_name=backbone,
        projection_dim=cfg['projection_dim'],
        pretrained_backbone=False,
        in_channels=in_channels,
    ).to(device)

    optimizer = AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    best_loss = float('inf')
    t0 = time.time()

    for epoch in range(1, cfg['epochs'] + 1):
        model.train()
        losses = []
        for batch in loader:
            x1, x2 = batch['image']
            x1, x2 = x1.to(device), x2.to(device)
            optimizer.zero_grad(set_to_none=True)
            _, z1 = model(x1)
            _, z2 = model(x2)
            loss = nt_xent_loss(z1, z2, temperature=cfg['temperature'])
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        epoch_loss = sum(losses) / max(len(losses), 1)
        if epoch % 10 == 0 or epoch == cfg['epochs']:
            elapsed = time.time() - t0
            print(f'  Epoch {epoch:03d}/{cfg["epochs"]:03d} | '
                  f'loss={epoch_loss:.4f} | {elapsed:.0f}s elapsed')
        if epoch_loss < best_loss:
            best_loss = epoch_loss
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'loss': best_loss,
                'config': {
                    'backbone_name': backbone,
                    'projection_dim': cfg['projection_dim'],
                    'temperature': cfg['temperature'],
                    'band_indices': band_indices,
                    'in_channels': in_channels,
                },
            }, best_pt)

    print(f'  Pretraining done. Best loss: {best_loss:.4f}')
    return str(best_pt)


def run_classification(exp_name, dataset_root, band_indices, backbone,
                       epochs, batch_size, lr, train_fraction, seed,
                       tail_epochs, image_size, device, output_dir,
                       checkpoint=None, freeze_encoder=False):
    """Runs one classification experiment. Returns results dict."""
    out = ensure_dir(output_dir)
    set_seed(seed)
    in_channels = len([int(x) for x in band_indices.split(',')])

    dataset = AssetClassificationDataset(
        dataset_root=dataset_root,
        transform=build_eval_transform(image_size),
        band_indices=band_indices,
    )
    classes = dataset.label_space.classes
    train_idxs, val_idxs = split_indices(len(dataset), train_fraction, seed)
    train_loader = DataLoader(Subset(dataset, train_idxs),
                              batch_size=batch_size, shuffle=True)
    val_loader   = DataLoader(Subset(dataset, val_idxs),
                              batch_size=batch_size, shuffle=False)

    # Build model
    if checkpoint and Path(checkpoint).exists():
        ckpt = torch.load(checkpoint, map_location='cpu')
        cfg  = ckpt.get('config', {}) if isinstance(ckpt, dict) else {}
        model = SimCLRModel(
            backbone_name=cfg.get('backbone_name', backbone),
            pretrained_backbone=False,
            projection_dim=cfg.get('projection_dim', 128),
            in_channels=in_channels,
        )
        sd = (ckpt.get('model_state') or ckpt.get('model_state_dict')
              or ckpt.get('state_dict') or ckpt)
        missing, _ = model.load_state_dict(sd, strict=False)
        encoder = model.backbone
        encoder.feature_dim = model.feature_dim
        print(f'  Loaded checkpoint ({len(sd)-len(missing)}/{len(sd)} keys matched)')
    else:
        encoder = EncoderBackbone(backbone, pretrained=False, in_channels=in_channels)
        print('  Random init encoder')

    head = LinearClassifier(encoder.feature_dim, len(classes))
    encoder.to(device); head.to(device)

    if freeze_encoder:
        for p in encoder.parameters():
            p.requires_grad = False

    params    = list(head.parameters()) + [p for p in encoder.parameters()
                                            if p.requires_grad]
    optimizer = AdamW(params, lr=lr, weight_decay=1e-4)
    weights   = compute_class_weights(dataset, train_idxs, device)
    criterion = nn.CrossEntropyLoss(weight=weights)

    best_acc    = -1.0
    val_history = []

    with (out / 'metrics.csv').open('w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=['epoch', 'train_loss', 'val_acc'])
        writer.writeheader()
        for epoch in range(1, epochs + 1):
            encoder.train(); head.train()
            losses = []
            for batch in train_loader:
                x = batch['image'].to(device)
                y = batch['label'].to(device)
                optimizer.zero_grad(set_to_none=True)
                loss = criterion(head(encoder(x)), y)
                loss.backward()
                optimizer.step()
                losses.append(loss.item())
            val_acc, _ = evaluate(encoder, head, val_loader, device)
            val_history.append(val_acc)
            train_loss = sum(losses) / max(len(losses), 1)
            writer.writerow({'epoch': epoch,
                             'train_loss': round(train_loss, 6),
                             'val_acc': round(val_acc, 6)})
            f.flush()
            if epoch % 10 == 0 or epoch == epochs:
                print(f'  Epoch {epoch:03d}/{epochs} | '
                      f'loss={train_loss:.4f} | val_acc={val_acc:.4f}')
            if val_acc > best_acc:
                best_acc = val_acc
                save_checkpoint(out / 'checkpoint_best.pt', {
                    'encoder_state': encoder.state_dict(),
                    'head_state':    head.state_dict(),
                    'classes':       classes,
                })

    tail_n    = min(tail_epochs, len(val_history))
    tail_vals = val_history[-tail_n:]
    tail_mean = sum(tail_vals) / len(tail_vals)
    tail_std  = (sum((v - tail_mean)**2 for v in tail_vals) / len(tail_vals))**0.5
    _, per_class = evaluate(encoder, head, val_loader, device, classes=classes)

    result = {
        'experiment':    exp_name,
        'best_val_acc':  round(best_acc, 4),
        'tail_mean_acc': round(tail_mean, 4),
        'tail_std_acc':  round(tail_std, 4),
        'tail_epochs':   tail_n,
        'per_class_acc': per_class,
        'classes':       classes,
        'freeze_encoder': freeze_encoder,
        'checkpoint':    checkpoint,
    }
    save_json(out / 'results_summary.json', result)
    return result


print('Utilities loaded.')

Utilities loaded.


In [8]:
import json
from pathlib import Path

GRID_DIR = '/content/drive/MyDrive/infra_fm/results/experiment_grid'

results = []
for p in sorted(Path(GRID_DIR).rglob('results_summary.json')):
    data = json.load(open(p))
    results.append(data)
    print(f"{data['experiment']:50s} best={data['best_val_acc']:.4f}  tail_mean={data['tail_mean_acc']:.4f}  tail_std={data['tail_std_acc']:.4f}")

random_init_finetuned                              best=0.6106  tail_mean=0.3361  tail_std=0.1161
ep100_t01_pd128_finetuned                          best=0.5966  tail_mean=0.3703  tail_std=0.0484
ep100_t01_pd128_linear_probe                       best=0.5462  tail_mean=0.4471  tail_std=0.0840
ep100_t02_pd128_finetuned                          best=0.5938  tail_mean=0.3445  tail_std=0.0806
ep100_t02_pd128_linear_probe                       best=0.5714  tail_mean=0.3725  tail_std=0.0775
ep100_t02_pd64_finetuned                           best=0.5798  tail_mean=0.3927  tail_std=0.0951
ep100_t02_pd64_linear_probe                        best=0.5546  tail_mean=0.2913  tail_std=0.0340
ep25_t02_pd128_finetuned                           best=0.5714  tail_mean=0.3989  tail_std=0.0497
ep25_t02_pd128_linear_probe                        best=0.5658  tail_mean=0.4095  tail_std=0.0761


## 6. Run experiment grid

This cell runs everything — pretraining then classification for each config.
Already-completed pretraining checkpoints are skipped automatically.
Expected total time on T4 GPU: ~2-3 hours for the full grid.

In [9]:
import time

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

all_results = []
grid_start  = time.time()

# --- Baseline: random init fine-tuned ---
print('\n' + '='*60)
print('BASELINE: random init fine-tuned')
print('='*60)
baseline = run_classification(
    exp_name       = 'random_init_finetuned',
    dataset_root   = DATASET_ROOT,
    band_indices   = BAND_INDICES,
    backbone       = BACKBONE,
    epochs         = CLASSIFY_EPOCHS,
    batch_size     = CLASSIFY_BATCH,
    lr             = CLASSIFY_LR,
    train_fraction = TRAIN_FRACTION,
    seed           = SEED,
    tail_epochs    = TAIL_EPOCHS,
    image_size     = IMAGE_SIZE,
    device         = device,
    output_dir     = f'{GRID_DIR}/baseline',
    checkpoint     = None,
    freeze_encoder = False,
)
all_results.append(baseline)
print(f'  -> tail_mean={baseline["tail_mean_acc"]:.4f} best={baseline["best_val_acc"]:.4f}')

# --- Grid: pretrain then classify ---
for cfg in PRETRAIN_GRID:
    pretrain_dir = f'{GRID_DIR}/pretrain_{cfg["name"]}'

    print(f'\n{"="*60}')
    print(f'PRETRAINING: {cfg["name"]}')
    print(f'  epochs={cfg["epochs"]}  temperature={cfg["temperature"]}  projection_dim={cfg["projection_dim"]}')
    print('='*60)

    checkpoint = run_pretraining(
        cfg          = cfg,
        dataset_root = DATASET_ROOT,
        band_indices = BAND_INDICES,
        n_optical    = N_OPTICAL,
        backbone     = BACKBONE,
        batch_size   = PRETRAIN_BATCH,
        lr           = PRETRAIN_LR,
        seed         = SEED,
        device       = device,
        output_dir   = pretrain_dir,
    )

    # Linear probe
    print(f'\nClassification: {cfg["name"]} — linear probe')
    result_probe = run_classification(
        exp_name       = f'{cfg["name"]}_linear_probe',
        dataset_root   = DATASET_ROOT,
        band_indices   = BAND_INDICES,
        backbone       = BACKBONE,
        epochs         = CLASSIFY_EPOCHS,
        batch_size     = CLASSIFY_BATCH,
        lr             = CLASSIFY_LR_PROBE,
        train_fraction = TRAIN_FRACTION,
        seed           = SEED,
        tail_epochs    = TAIL_EPOCHS,
        image_size     = IMAGE_SIZE,
        device         = device,
        output_dir     = f'{GRID_DIR}/classify_{cfg["name"]}_probe',
        checkpoint     = checkpoint,
        freeze_encoder = True,
    )
    all_results.append(result_probe)
    print(f'  -> tail_mean={result_probe["tail_mean_acc"]:.4f} best={result_probe["best_val_acc"]:.4f}')

    # Fine-tuned
    print(f'\nClassification: {cfg["name"]} — fine-tuned')
    result_ft = run_classification(
        exp_name       = f'{cfg["name"]}_finetuned',
        dataset_root   = DATASET_ROOT,
        band_indices   = BAND_INDICES,
        backbone       = BACKBONE,
        epochs         = CLASSIFY_EPOCHS,
        batch_size     = CLASSIFY_BATCH,
        lr             = CLASSIFY_LR,
        train_fraction = TRAIN_FRACTION,
        seed           = SEED,
        tail_epochs    = TAIL_EPOCHS,
        image_size     = IMAGE_SIZE,
        device         = device,
        output_dir     = f'{GRID_DIR}/classify_{cfg["name"]}_finetuned',
        checkpoint     = checkpoint,
        freeze_encoder = False,
    )
    all_results.append(result_ft)
    print(f'  -> tail_mean={result_ft["tail_mean_acc"]:.4f} best={result_ft["best_val_acc"]:.4f}')

total_time = time.time() - grid_start
print(f'\nGrid complete in {total_time/60:.1f} minutes.')

# Save all results
with open(f'{GRID_DIR}/all_results.json', 'w') as f:
    json.dump(all_results, f, indent=2)
print(f'Saved all results to {GRID_DIR}/all_results.json')

Device: cpu

BASELINE: random init fine-tuned


KeyboardInterrupt: 

## 6.5. Remaining pretraining/classification after GPU crash

In [ ]:
device = torch.device('cpu')
print('Running on CPU — just 2 remaining classification runs')

checkpoint_pd64 = f'{GRID_DIR}/pretrain_ep100_t02_pd64/best.pt'

result_probe_pd64 = run_classification(
    exp_name       = 'ep100_t02_pd64_linear_probe',
    dataset_root   = DATASET_ROOT,
    band_indices   = BAND_INDICES,
    backbone       = BACKBONE,
    epochs         = CLASSIFY_EPOCHS,
    batch_size     = CLASSIFY_BATCH,
    lr             = CLASSIFY_LR_PROBE,
    train_fraction = TRAIN_FRACTION,
    seed           = SEED,
    tail_epochs    = TAIL_EPOCHS,
    image_size     = IMAGE_SIZE,
    device         = device,
    output_dir     = f'{GRID_DIR}/classify_ep100_t02_pd64_probe',
    checkpoint     = checkpoint_pd64,
    freeze_encoder = True,
)
print(f'probe -> tail_mean={result_probe_pd64["tail_mean_acc"]:.4f}')

result_ft_pd64 = run_classification(
    exp_name       = 'ep100_t02_pd64_finetuned',
    dataset_root   = DATASET_ROOT,
    band_indices   = BAND_INDICES,
    backbone       = BACKBONE,
    epochs         = CLASSIFY_EPOCHS,
    batch_size     = CLASSIFY_BATCH,
    lr             = CLASSIFY_LR,
    train_fraction = TRAIN_FRACTION,
    seed           = SEED,
    tail_epochs    = TAIL_EPOCHS,
    image_size     = IMAGE_SIZE,
    device         = device,
    output_dir     = f'{GRID_DIR}/classify_ep100_t02_pd64_finetuned',
    checkpoint     = checkpoint_pd64,
    freeze_encoder = False,
)
print(f'finetuned -> tail_mean={result_ft_pd64["tail_mean_acc"]:.4f}')

## 7. Comparison table

In [11]:
import json
from pathlib import Path

GRID_DIR = '/content/drive/MyDrive/infra_fm/results/experiment_grid'

# Load all completed results
results = []
for p in sorted(Path(GRID_DIR).rglob('results_summary.json')):
    data = json.load(open(p))
    results.append(data)

# Save the combined file so we have it going forward
with open(f'{GRID_DIR}/all_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print(f'Loaded {len(results)} experiments, saved to all_results.json\n')

# Print comparison table
classes = results[0]['classes']
def short_class(c):
    return c.replace('energy.','').replace('distribution.','dx_').replace('transmission.','tx_')
short_classes = [short_class(c) for c in classes]

print('='*90)
print('EXPERIMENT GRID — COMPARISON TABLE')
print('='*90)
print(f'{"Experiment":50s} {"Best":>6} {"Tail mean":>10} {"Tail std":>10}')
print('-'*90)
for r in sorted(results, key=lambda x: x['tail_mean_acc'], reverse=True):
    name = r['experiment'].replace('_', ' ')
    marker = ' *' if r.get('freeze_encoder') else ''
    print(f'{name+marker:50s} {r["best_val_acc"]:>6.4f} {r["tail_mean_acc"]:>10.4f} {r["tail_std_acc"]:>10.4f}')
print('(* = linear probe / frozen encoder)')

print(f'\nPer-class accuracy (final epoch):')
col_w = 20
print(f'{"":50s}' + ''.join(f'{c:>{col_w}}' for c in short_classes))
print('-'*(50 + col_w*len(classes)))
for r in sorted(results, key=lambda x: x['tail_mean_acc'], reverse=True):
    name = r['experiment'].replace('_', ' ')
    marker = ' *' if r.get('freeze_encoder') else ''
    row = f'{name+marker:50s}'
    if r.get('per_class_acc'):
        for cls in classes:
            row += f'{r["per_class_acc"].get(cls, 0.0):>{col_w}.4f}'
    print(row)

best = max(results, key=lambda r: r['tail_mean_acc'])
print(f'\nBest tail mean: {best["experiment"]} — {best["tail_mean_acc"]:.4f}')

Loaded 9 experiments, saved to all_results.json

EXPERIMENT GRID — COMPARISON TABLE
Experiment                                           Best  Tail mean   Tail std
------------------------------------------------------------------------------------------
ep100 t01 pd128 linear probe *                     0.5462     0.4471     0.0840
ep25 t02 pd128 linear probe *                      0.5658     0.4095     0.0761
ep25 t02 pd128 finetuned                           0.5714     0.3989     0.0497
ep100 t02 pd64 finetuned                           0.5798     0.3927     0.0951
ep100 t02 pd128 linear probe *                     0.5714     0.3725     0.0775
ep100 t01 pd128 finetuned                          0.5966     0.3703     0.0484
ep100 t02 pd128 finetuned                          0.5938     0.3445     0.0806
random init finetuned                              0.6106     0.3361     0.1161
ep100 t02 pd64 linear probe *                      0.5546     0.2913     0.0340
(* = linear probe / froze